<a href="https://colab.research.google.com/github/nvthz/conecta-cultura/blob/main/Projeto_conecta_cultura_%7C_Imers%C3%A3o_Alura_e_Google_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install google-genai

In [ ]:
# Configura a API Key do Google Gemini

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
# Configura o cliente da SDK do Gemini

from google import genai

client = genai.Client()

MODEL_ID = "gemini-2.5-pro-preview-03-25"

In [ ]:
# Pergunta ao Gemini uma informação mais recente que seu conhecimento

from IPython.display import HTML, Markdown

# Perguntar pro modelo quando é a próxima imersão de IA ###############################################
resposta = client.models.generate_content(
    model=MODEL_ID,
    contents='Quando é a próxima Imersão IA com Google Gemini da Alura?',
)

# Exibe a resposta na tela
display(Markdown(f"Resposta:\n {resposta.text}"))

Resposta:
 A Alura não tem uma data fixa para a Imersão IA com Google Gemini. A melhor forma de saber quando a próxima edição será realizada é:

*   **Acompanhar as redes sociais da Alura:** Eles geralmente anunciam novas edições de cursos e imersões no Instagram, LinkedIn, Twitter e Facebook.
*   **Verificar a página de Imersões da Alura:** Acesse o site da Alura e procure pela seção de Imersões. Lá, você poderá encontrar informações sobre as imersões futuras e a possibilidade de se inscrever para ser notificado.
*   **Assinar a newsletter da Alura:** Ao assinar a newsletter, você receberá informações sobre lançamentos, promoções e eventos, incluindo as Imersões.

In [ ]:
# Exibe a busca
print(f"Busca realizada: {response.candidates[0].grounding_metadata.web_search_queries}")
# Exibe as URLs nas quais ele se baseou
print(f"Páginas utilizadas na resposta: {', '.join([site.web.title for site in response.candidates[0].grounding_metadata.grounding_chunks])}")
print()
display(HTML(response.candidates[0].grounding_metadata.search_entry_point.rendered_content))

Busca realizada: ['Alura Imersão IA com Google Gemini']
Páginas utilizadas na resposta: alura.com.br, tecmundo.com.br



In [ ]:
# Instalar Framework ADK de agentes do Google ################################################
!pip install -q google-adk

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types  # Para criar conteúdos (Content e Part)
from datetime import date
import textwrap # Para formatar melhor a saída de texto
from IPython.display import display, Markdown # Para exibir texto formatado no Colab
import requests # Para fazer requisições HTTP
import warnings

warnings.filterwarnings("ignore")

In [69]:
# Função auxiliar que envia uma mensagem para um agente via Runner e retorna a resposta final
def call_agent(agent: Agent, message_text: str) -> str:
    # Cria um serviço de sessão em memória
    session_service = InMemorySessionService()
    # Cria uma nova sessão (você pode personalizar os IDs conforme necessário)
    session = session_service.create_session(app_name=agent.name, user_id="user1", session_id="session1")
    # Cria um Runner para o agente
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
    # Cria o conteúdo da mensagem de entrada
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""
    # Itera assincronamente pelos eventos retornados durante a execução do agente
    for event in runner.run(user_id="user1", session_id="session1", new_message=content):
        if event.is_final_response():
          for part in event.content.parts:
            if part.text is not None:
              final_response += part.text
              final_response += "\n"
    return final_response

In [ ]:
# Função auxiliar para exibir texto formatado em Markdown no Colab
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [ ]:
!pip install -q -U google-generativeai

In [110]:
##########################################
# --- Agente 1: Buscador de Eventos --- #
##########################################
def agente_buscador(topico, data_de_hoje):
  buscador = Agent(
      name="agente_buscador",
      model="gemini-2.0-flash",
      instruction="""
Você é um agente de busca. Sua missão é encontrar os 5 eventos culturais mais interessantes acontecendo na cidade de [NOME DA CIDADE] nos próximos 7 dias. Dê preferência a eventos como:

* Exposições
* Oficinas
* Mostras culturais
* Apresentações musicais
* Programações do SESC, centros culturais ou museus

Para cada evento encontrado, siga estes passos:

1.  **Busque informações detalhadas sobre o evento no Google.**
2.  **Procure ativamente por 1 ou 2 links de redes sociais (TikTok, Instagram, YouTube Shorts ou Reels) que mostrem o evento, a experiência ou bastidores.** Priorize links de conteúdo popular e com boas críticas.

Para cada evento, inclua:

* Título
* Local
* Data e horário
* Se é gratuito ou pago
* Uma descrição curta
* Link da fonte principal
* Informação adicional sobre o que cada link de rede social mostra (ex: "Este vídeo no TikTok mostra a fila animada na entrada", "Este Reel do Instagram dá um tour rápido pela exposição").
* Os 1 ou 2 links de redes sociais encontrados.
      """,
      description="Agente que busca informações no Google",
      tools=[google_search],
  )

  entrada_do_agente_buscador = f"Tópico: {topico}\nData de hoje: {data_de_hoje}"

  novos_eventos = call_agent(buscador, entrada_do_agente_buscador)
  return novos_eventos

In [111]:
################################################
# --- Agente 2: Planejador de cultura --- #
################################################
def agente_planejador(topico, novos_eventos_buscados):
    planejador = Agent(
        name="agente_planejador",
        model="gemini-2.0-flash",
        # Inserir as instruções do Agente Planejador #################################################
        instruction="""
       Você é um agente planejador com olhar cultural e sensível. Sua função é escolher 3 eventos da lista fornecida que:

* Tenham temas que toquem pessoas reais, como identidade, memória, afeto, pertencimento (eventos que ativam pertencimento étnico, cultural, ancestralidade; inspiram união de natureza, arte e comunidade; criam pertencimento por vivência compartilhada; reforçam acolhimento e voz; ativam memória afetiva, intimidade, conexão com sentimentos humanos).
* Ofereçam experiências visuais ou sensoriais marcantes, com potencial de serem capturadas em vídeos ou imagens impactantes (vídeos que mostram a energia do evento, interações com instalações visuais, cenas emocionantes, etc.).
* Sejam populares e tenham boas críticas, indicando um forte engajamento do público jovem (18-35 anos) com interesse em mídias sociais e tendências atuais.
* Tenham forte potencial de despertar inspiração ou pertencimento.

Para cada um dos 3 eventos selecionados, organize assim:

* Título do evento
* Tema principal (ex: arte urbana, cultura popular, introspecção, ancestralidade, etc.)
* Justificativa de por que ele foi escolhido, com foco no potencial de despertar inspiração ou pertencimento e no apelo para o público jovem.
* Destaque visual, se houver (mencione qualquer imagem, vídeo ou cena marcante que possa ser um "gancho sensorial", como: “vídeo incrível no TikTok mostra a entrada da exposição”, “Tem um vídeo em que as pessoas interagem com espelhos gigantes no meio da praça”, “Um Reel mostra crianças dançando no meio da exposição, é lindo de ver”, “O mural de entrada já é uma obra de arte à parte — tem vídeo no Instagram mostrando o processo”).
        """,
        description="Agente que planeja cultura",
        tools=[google_search]
    )

    entrada_do_agente_planejador = f"Tópico:{topico}\nNovos eventos buscados: {novos_eventos_buscados}"
    # Executa o agente
    plano_de_cultura = call_agent(planejador, entrada_do_agente_planejador)
    return plano_de_cultura

In [112]:
######################################
# --- Agente 3: Refinador de cultura --- #
######################################
def agente_refinador(topico, plano_de_cultura):
    redator = Agent(
        name="agente_refinador",
        model="gemini-2.0-flash",
        instruction="""
            Você é um redator afetuoso e criativo. Sua missão é transformar os eventos selecionados em **posts** envolventes que gerem interesse e vontade de participar no usuário. O tom deve ser leve, inspirador e próximo, como se estivesse indicando algo especial para uma amiga.

Para cada evento:

* Escreva um parágrafo de apresentação sensível que conecte o usuário com o tema do evento.
* Inclua título, local, data e valor.
* Inclua os links das redes sociais com frases que incentivem a visualização:
    * “Dá uma olhada nesse vídeo no TikTok pra sentir a energia do lugar!”
    * “Confere esse Reel no Instagram, mostra um pouquinho da experiência!”
* Finalize com uma frase de convite/reflexão que motive o usuário a considerar o evento.
            """,
        description="Agente refinador de eventos populares nas redes sociais"
    )
    entrada_do_agente_refinador = f"Tópico: {topico}\nPlano de cultura: {plano_de_cultura}"
    # Executa o agente
    refine = call_agent(redator, entrada_do_agente_refinador)
    return refine

In [113]:
##########################################
# --- Agente 4: Revisor de Qualidade --- #
##########################################
def agente_revisor(topico, refine_gerado):
    revisor = Agent(
        name="agente_revisor",
        model="gemini-2.0-flash",
        instruction="""
       Você é um revisor com olhar poético e acolhedor. Sua tarefa é revisar os **posts** produzidos pelo redator garantindo:

* Clareza e correção gramatical.
* Tom sensível, convidativo e fluido, adequado para um post de recomendação.
* Uniformidade na estrutura dos eventos nos posts.
* Delicadeza na escolha de palavras e frases, visando gerar conexão e engajamento no usuário.

Se achar necessário, suavize expressões e proponha melhorias que tornem o post ainda mais inspirador e convidativo. Ao final, organize os 3 eventos em um formato de **posts** separados, com espaçamento adequado para leitura digital e elementos que facilitem o engajamento (como quebras de linha e formatação).
            """,
        description="Agente revisor de eventos populares nas redes sociais."
    )
    entrada_do_agente_revisor = f"Tópico: {topico}\nRefine: {refine_gerado}"
    # Executa o agente
    evento_revisado = call_agent(revisor, entrada_do_agente_revisor)
    return evento_revisado

In [ ]:
data_de_hoje = date.today().strftime("%d/%m/%Y")

print("🎭 Iniciando o Sistema de Recomendação de Eventos Culturais com 4 Agentes 🎭")

# --- Obter o Tópico do Usuário ---
topico = input("❓ Qual cidade você quer explorar hoje? ")

# Inserir lógica do sistema de agentes ################################################
if not topico:
  print("Você esqueceu de digitar o tópico!")
else:
  print(f"Legal! Vamos ver os eventos que estão rolando por aí...")


  novos_eventos_buscados = agente_buscador(topico, data_de_hoje)
  print("\n--- 📝 Resultado do Agente 1 (Buscador) ---\n")
  display(to_markdown(novos_eventos_buscados))
  print("-----------------------------------------------------------")

  plano_de_cultura = agente_planejador(topico, novos_eventos_buscados)
  print("\n--- 📝 Resultado do Agente 2 (Planejador) ---\n")
  display(to_markdown(plano_de_cultura))
  print("-----------------------------------------------------------")

  refine_gerado = agente_refinador(topico, plano_de_cultura)
  print("\n--- 📝 Resultado do Agente 3 (Refinador) ---\n")
  display(to_markdown(refine_de_evento))
  print("-----------------------------------------------------------")

  eventos_finais = agente_revisor(topico, refine_gerado)
  print("\n--- 📝 Resultado do Agente 4 (Revisor) ---\n")
  display(to_markdown(eventos_finais))
  print("-----------------------------------------------------------")

🎭 Iniciando o Sistema de Recomendação de Eventos Culturais com 4 Agentes 🎭
